# AAU-PII — Adaptive Adversarial Unlearning Results Analysis

Notebook đọc trực tiếp từ `aggregate_metrics.json` trong các thư mục `eval_results/` trên Google Drive.

**AAU-PII variants được phân tích:**
- **AAU-GA / ep5 warm-start** — warm-start từ GA+Retain 5ep, inner method GA (v2, v3)
- **AAU-GA / ep3 warm-start** — warm-start từ GA+Retain 3ep, inner method GA
- **AAU-NPO v1** — warm-start từ NPO rw=2 ep3, inner method NPO (rw=2, 50 steps)
- **AAU-NPO v2** — warm-start từ NPO rw=2 ep3, inner method NPO (rw=4, 25 steps)
- **AAU-TV / Round 3** — warm-start từ Task Vector α=2, inner method GA, checkpoint round 3
- **AAU-TV / Final** — warm-start từ Task Vector α=2, inner method GA, checkpoint cuối

**Phần tổng kết:** So sánh AAU-PII với các baseline methods tốt nhất.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib
from scipy.stats import hmean

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 120
plt.style.use('seaborn-v0_8-whitegrid')
print('[OK] Ready')

In [ ]:
# ======================================================================
# CONFIGURATION
# ======================================================================

BASE_DIR = '/content/drive/MyDrive/pii-unlearning_saved_checkpoints/UnlearnPII'

# --- AAU-PII checkpoints (display name -> relative path from BASE_DIR) ---
AAU_CHECKPOINTS = {
    # GA inner method, ep5 warm-start
    'AAU-GA ep5 v2':        'pii_unlearn_aau_pii_grad_diff_5r_v2',
    'AAU-GA ep5 v3':        'pii_unlearn_aau_pii_grad_diff_5r_v3',
    # GA inner method, ep3 warm-start
    'AAU-GA ep3':           'pii_unlearn_aau_pii_grad_diff_3ep_v1',
    # NPO inner method, NPO rw=2 ep3 warm-start
    'AAU-NPO v1':           'pii_unlearn_aau_pii_npo_rw2_ep3_v1',
    'AAU-NPO v2':           'pii_unlearn_aau_pii_npo_rw2_ep3_v2',
    # GA inner method, Task Vector α=2 warm-start
    'AAU-TV Round3':        'pii_unlearn_aau_pii_tv_alpha2_v1/round_3_step21',
    'AAU-TV Final':         'pii_unlearn_aau_pii_tv_alpha2_v1',
}

# --- Baseline checkpoints for summary comparison ---
BASELINE_CHECKPOINTS = {
    'SFT Exposed':          'sft_exposed_llama2-7b_r64',
    'GA ep3':               'pii_unlearn_ga_only_3ep',
    'GA+Retain ep3':        'pii_unlearn_grad_diff_5ep/checkpoint-step21',
    'GA+Retain ep5':        'pii_unlearn_grad_diff_5ep',
    'Task Vector α=2':      'pii_unlearn_tv_alpha2_8ep',
    'NPO rw=2 ep3':         'pii_unlearn_npo_rw2_5ep/checkpoint-step21',
    'NPO rw=1 ep5':         'pii_unlearn_npo_rw1_5ep',
    'NPO rw=2 ep5':         'pii_unlearn_npo_rw2_5ep',
}

# Method order for display
AAU_ORDER = list(AAU_CHECKPOINTS.keys())
BASELINE_ORDER = list(BASELINE_CHECKPOINTS.keys())

# Colors for AAU methods
AAU_COLORS = {
    'AAU-GA ep5 v2':   '#1f77b4',
    'AAU-GA ep5 v3':   '#aec7e8',
    'AAU-GA ep3':      '#2ca02c',
    'AAU-NPO v1':      '#d62728',
    'AAU-NPO v2':      '#ff9896',
    'AAU-TV Round3':   '#9467bd',
    'AAU-TV Final':    '#c5b0d5',
}

# Markers for scatter
AAU_MARKERS = {
    'AAU-GA ep5 v2':   'o',
    'AAU-GA ep5 v3':   's',
    'AAU-GA ep3':      'D',
    'AAU-NPO v1':      '^',
    'AAU-NPO v2':      'v',
    'AAU-TV Round3':   'P',
    'AAU-TV Final':    'X',
}

print(f'[OK] {len(AAU_CHECKPOINTS)} AAU variants, {len(BASELINE_CHECKPOINTS)} baselines configured')

In [ ]:
# ======================================================================
# DATA LOADING — reads individual task JSON files, same as v2 notebook
# ======================================================================

TASK_FILES = [
    'eval_log_forget',
    'eval_log_forget_paraphrase_1',
    'eval_log_forget_paraphrase_2',
    'eval_log_forget_paraphrase_3',
    'eval_log_forget_paraphrase_4',
    'eval_log_forget_paraphrase_5',
    'eval_log_forget_inverse',
    'one_hop_attack',
    'targeted_extraction',
    'eval_log_retain',
    'eval_real_author',
    'eval_real_world',
]


def get_display_name(task_name):
    """Map task file name -> display name (same logic as evaluate.py)."""
    if 'inverse' in task_name:
        return 'Forget Inverse'
    if 'paraphrase' in task_name or 'rephrase' in task_name:
        if 'forget' in task_name:
            return 'Forget Rephrase'
        return 'Retain Rephrase'
    if 'forget' in task_name:
        return 'Forget'
    if 'one_hop' in task_name:
        return 'One-Hop'
    if 'targeted_extraction' in task_name:
        return 'Targeted Extraction'
    if 'retain' in task_name:
        return 'Retain'
    if 'real_world' in task_name:
        return 'Real World'
    if 'real_author' in task_name:
        return 'Real Authors'
    return task_name


def load_task_data(checkpoint_path):
    """Load all task JSON files from a checkpoint's eval_results/."""
    eval_dir = os.path.join(checkpoint_path, 'eval_results')
    if not os.path.isdir(eval_dir):
        return {}
    task_data = {}
    for task in TASK_FILES:
        fpath = os.path.join(eval_dir, f'{task}.json')
        if os.path.exists(fpath):
            with open(fpath) as f:
                task_data[task] = json.load(f)
    return task_data


def compute_aggregate(task_data):
    """Recompute aggregate metrics from raw task JSONs.
    Mirrors evaluate.py compute_aggregate_metrics() exactly.
    """
    collected = {}

    def _add(key, val):
        collected.setdefault(key, []).append(val)

    for task_name, logs in task_data.items():
        display = get_display_name(task_name)

        # Targeted extraction: special keys
        if 'targeted_extraction_forget_esr' in logs:
            _add('Targeted Extraction Forget ESR', logs['targeted_extraction_forget_esr'])
            if 'targeted_extraction_retain_esr' in logs:
                _add('Targeted Extraction Retain ESR', logs['targeted_extraction_retain_esr'])
            if 'pii_leakage_rate' in logs:
                _add(f'PII Leakage {display}', logs['pii_leakage_rate'])
            continue

        # Probability
        if 'avg_gt_loss' in logs:
            if 'eval_log' in task_name or 'one_hop' in task_name:
                gt_probs = np.exp(-1 * np.array(list(logs['avg_gt_loss'].values()), dtype=np.float64))
                _add(f'Prob. {display}', float(np.mean(gt_probs)))
            elif 'average_perturb_loss' in logs:
                avg_true  = np.exp(-1 * np.array(list(logs['avg_gt_loss'].values()), dtype=np.float64))
                avg_false = np.exp(-1 * np.array(list(logs['average_perturb_loss'].values()), dtype=np.float64))
                avg_all   = np.concatenate([np.expand_dims(avg_true, axis=-1), avg_false], axis=1).sum(-1)
                _add(f'Prob. {display}', float(np.mean(avg_true / avg_all)))
            else:
                gt_probs = np.exp(-1 * np.array(list(logs['avg_gt_loss'].values()), dtype=np.float64))
                _add(f'Prob. {display}', float(np.mean(gt_probs)))

        # ROUGE-L
        if 'rougeL_recall' in logs:
            _add(f'ROUGE {display}', float(np.mean(list(logs['rougeL_recall'].values()))))

        # Truth Ratio
        if 'avg_paraphrased_loss' in logs and 'average_perturb_loss' in logs:
            para_vals    = np.array(list(logs['avg_paraphrased_loss'].values()), dtype=np.float64)
            perturb_vals = np.array(list(logs['average_perturb_loss'].values()), dtype=np.float64)
            perturb_mean = perturb_vals.mean(axis=-1) if perturb_vals.ndim > 1 else perturb_vals
            curr_stat    = np.exp(perturb_mean - para_vals)
            if 'forget' in task_name:
                tr = float(np.mean(np.minimum(curr_stat, 1 / curr_stat)))
            else:
                tr = float(np.mean(np.maximum(0, 1 - 1 / curr_stat)))
            _add(f'Truth Ratio {display}', tr)

        # Fluency
        if 'fluency' in logs:
            _add(f'Fluency {display}', logs['fluency'])

        # PII Leakage
        if 'pii_leakage_rate' in logs:
            _add(f'PII Leakage {display}', logs['pii_leakage_rate'])

    # Average collected values (5 paraphrase tasks -> single 'Forget Rephrase')
    output = {}
    for key, vals in collected.items():
        output[key] = float(np.mean(vals))

    # Model Utility: hmean of retain + general knowledge metrics
    UTILITY_EXCLUDE = ('Forget', 'Rephrase', 'Fluency', 'Inverse',
                       'PII Leakage', 'ESR', 'One-Hop', 'Extraction')
    utility_cands = [v for k, v in output.items()
                     if not any(excl in k for excl in UTILITY_EXCLUDE)
                     and isinstance(v, (int, float)) and v > 0]
    if utility_cands:
        output['Model Utility'] = float(hmean(utility_cands))

    return output


def load_all(checkpoint_dict):
    """Load and compute aggregate metrics for all checkpoints."""
    results = {}
    for name, rel_path in checkpoint_dict.items():
        full_path = os.path.join(BASE_DIR, rel_path)
        task_data = load_task_data(full_path)
        if task_data:
            results[name] = compute_aggregate(task_data)
            print(f'  [OK] {name}  ({len(task_data)} task files)')
        else:
            print(f'  [SKIP] {name} — eval_results not found at {full_path}')
    return results


print('Loading AAU-PII results...')
aau_data = load_all(AAU_CHECKPOINTS)
print(f'\nLoading baseline results...')
baseline_data = load_all(BASELINE_CHECKPOINTS)
print(f'\n[OK] Loaded {len(aau_data)} AAU variants, {len(baseline_data)} baselines')

In [ ]:
# ======================================================================
# BUILD DATAFRAMES
# ======================================================================

# Key metrics to display (matching aggregate_metrics.json keys)
KEY_METRICS = [
    'Prob. Forget',
    'ROUGE Forget',
    'PII Leakage Forget',
    'PII Leakage Forget Rephrase',
    'PII Leakage Forget Inverse',
    'PII Leakage One-Hop',
    'Targeted Extraction Forget ESR',
    'Targeted Extraction Retain ESR',
    'Prob. Retain',
    'Truth Ratio Retain',
    'PII Leakage Retain',
    'Model Utility',
]

def build_df(data_dict, order):
    rows = {}
    for name in order:
        if name in data_dict:
            rows[name] = {k: data_dict[name].get(k, np.nan) for k in KEY_METRICS}
    return pd.DataFrame(rows).T

aau_df = build_df(aau_data, AAU_ORDER)
baseline_df = build_df(baseline_data, BASELINE_ORDER)

print('AAU-PII DataFrame:')
print(aau_df.to_string(float_format='{:.4f}'.format))

In [ ]:
# ======================================================================
# HEATMAP — AAU-PII metrics
# ======================================================================

# Metrics where LOWER is better (for forget quality)
LOWER_BETTER = {
    'Prob. Forget', 'ROUGE Forget',
    'PII Leakage Forget', 'PII Leakage Forget Rephrase', 'PII Leakage Forget Inverse',
    'PII Leakage One-Hop',
    'Targeted Extraction Forget ESR', 'Targeted Extraction Retain ESR',
    'PII Leakage Retain',
}

def make_heatmap(df, title, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(16, max(3, len(df) * 0.7)))
    else:
        fig = ax.get_figure()

    n_rows, n_cols = df.shape
    norm_df = df.copy()

    for col in df.columns:
        col_vals = df[col].dropna()
        if col_vals.empty:
            norm_df[col] = 0.5
            continue
        lo, hi = col_vals.min(), col_vals.max()
        if hi == lo:
            norm_df[col] = 0.5
        else:
            normed = (df[col] - lo) / (hi - lo)
            # Invert so green = better
            norm_df[col] = normed if col not in LOWER_BETTER else 1 - normed

    cmap = plt.cm.RdYlGn
    img = ax.imshow(norm_df.values.astype(float), cmap=cmap, aspect='auto',
                    vmin=0, vmax=1)

    # Labels
    ax.set_xticks(range(n_cols))
    ax.set_xticklabels(df.columns, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(n_rows))
    ax.set_yticklabels(df.index, fontsize=9)

    # Value annotations
    for i in range(n_rows):
        for j in range(n_cols):
            val = df.iloc[i, j]
            if not np.isnan(val):
                bg = norm_df.iloc[i, j]
                txt_col = 'black' if 0.3 < bg < 0.7 else ('white' if bg < 0.3 or bg > 0.85 else 'black')
                ax.text(j, i, f'{val:.3f}', ha='center', va='center',
                        fontsize=7.5, color=txt_col, fontweight='bold')

    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
    plt.colorbar(img, ax=ax, label='Normalized score (green=better)', shrink=0.8)
    return fig

fig = make_heatmap(aau_df, 'AAU-PII — Key Metrics Heatmap (green = better)')
plt.tight_layout()
plt.show()

In [ ]:
# ======================================================================
# PARETO SCATTER — Forget vs Utility (AAU-PII only)
# ======================================================================

fig, ax = plt.subplots(figsize=(9, 6))

sft_util  = baseline_data.get('SFT Exposed', {}).get('Model Utility', None)
sft_leak  = baseline_data.get('SFT Exposed', {}).get('PII Leakage Forget', None)

if sft_util:
    ax.axhline(sft_util, color='gray', ls='--', lw=1, alpha=0.7, label=f'SFT Utility={sft_util:.3f}')
if sft_leak:
    ax.axvline(1 - sft_leak, color='gray', ls=':', lw=1, alpha=0.7, label=f'SFT Forget={sft_leak:.3f}')

for name in AAU_ORDER:
    if name not in aau_data:
        continue
    m = aau_data[name]
    x = 1 - m.get('PII Leakage Forget', np.nan)  # higher = better forget
    y = m.get('Model Utility', np.nan)
    ax.scatter(x, y, color=AAU_COLORS[name], marker=AAU_MARKERS[name],
               s=120, zorder=5, edgecolors='black', linewidths=0.6)
    ax.annotate(name, (x, y), textcoords='offset points', xytext=(6, 4),
                fontsize=7.5, color=AAU_COLORS[name], fontweight='bold')

ax.set_xlabel('Forget Quality  (1 − PII Leakage Forget, higher = better)', fontsize=10)
ax.set_ylabel('Model Utility (higher = better)', fontsize=10)
ax.set_title('AAU-PII — Forget–Utility Pareto Scatter', fontsize=12, fontweight='bold')
ax.legend(fontsize=8)

# Annotate ideal region
ax.annotate('← Ideal region →', xy=(0.92, 0.55), fontsize=8, color='green',
            style='italic', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
# ======================================================================
# BAR CHARTS — Key metric comparison across AAU variants
# ======================================================================

plot_metrics = [
    ('PII Leakage Forget',           'PII Leakage — Direct',         True),
    ('PII Leakage Forget Rephrase',  'PII Leakage — Rephrase',       True),
    ('Targeted Extraction Forget ESR','Targeted Extraction ESR',      True),
    ('Model Utility',                 'Model Utility',                False),
    ('Prob. Retain',                  'Prob. Retain',                 False),
    ('Truth Ratio Retain',            'Truth Ratio Retain',           False),
]

methods = [m for m in AAU_ORDER if m in aau_data]
x = np.arange(len(methods))
width = 0.65

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for idx, (metric, label, lower_better) in enumerate(plot_metrics):
    ax = axes[idx]
    vals = [aau_data[m].get(metric, np.nan) for m in methods]
    colors = [AAU_COLORS[m] for m in methods]
    bars = ax.bar(x, vals, width, color=colors, edgecolor='black', linewidth=0.5)

    for bar, val in zip(bars, vals):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=7)

    ax.set_xticks(x)
    ax.set_xticklabels([m.replace(' ', '\n') for m in methods], fontsize=7)
    ax.set_title(f'{label}\n({"↓ lower better" if lower_better else "↑ higher better"})',
                 fontsize=9, fontweight='bold')
    ax.set_ylim(0, min(1.15, max([v for v in vals if not np.isnan(v)], default=1) * 1.2))

    # SFT reference line for forget metrics
    if 'SFT Exposed' in baseline_data and metric in baseline_data['SFT Exposed']:
        sft_val = baseline_data['SFT Exposed'][metric]
        ax.axhline(sft_val, color='red', ls='--', lw=1, alpha=0.6,
                   label=f'SFT={sft_val:.3f}')
        ax.legend(fontsize=7)

fig.suptitle('AAU-PII — Key Metrics by Variant', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ======================================================================
# GROUPING ANALYSIS — Warm-start & Inner method
# ======================================================================

groups = {
    'GA inner / ep5 warm':  ['AAU-GA ep5 v2', 'AAU-GA ep5 v3'],
    'GA inner / ep3 warm':  ['AAU-GA ep3'],
    'NPO inner / NPO warm': ['AAU-NPO v1', 'AAU-NPO v2'],
    'GA inner / TV warm':   ['AAU-TV Round3', 'AAU-TV Final'],
}

summary_metrics = ['PII Leakage Forget', 'Model Utility', 'Targeted Extraction Forget ESR']

print('=' * 70)
print(f'  {"Method":<22}  {"PII Leak":>10}  {"Utility":>10}  {"ESR":>10}')
print('=' * 70)

for group_name, members in groups.items():
    print(f'\n  [{group_name}]')
    for name in members:
        if name not in aau_data:
            continue
        m = aau_data[name]
        pii  = m.get('PII Leakage Forget', np.nan)
        util = m.get('Model Utility', np.nan)
        esr  = m.get('Targeted Extraction Forget ESR', np.nan)
        print(f'  {name:<24} {pii:>10.4f}  {util:>10.4f}  {esr:>10.4f}')

print('=' * 70)

In [ ]:
# ======================================================================
# RADAR CHART — AAU-PII multi-dimensional comparison
# ======================================================================

radar_metrics = [
    ('1 - PII Leak\n(Direct)',    lambda m: 1 - m.get('PII Leakage Forget', 1)),
    ('1 - PII Leak\n(Rephrase)',  lambda m: 1 - m.get('PII Leakage Forget Rephrase', 1)),
    ('1 - ESR\n(Forget)',         lambda m: 1 - m.get('Targeted Extraction Forget ESR', 1)),
    ('Prob.\nRetain',             lambda m: m.get('Prob. Retain', 0)),
    ('Truth Ratio\nRetain',       lambda m: m.get('Truth Ratio Retain', 0)),
    ('Model\nUtility',            lambda m: m.get('Model Utility', 0)),
]

labels = [r[0] for r in radar_metrics]
N = len(labels)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

# SFT reference
if 'SFT Exposed' in baseline_data:
    sft_vals = [fn(baseline_data['SFT Exposed']) for _, fn in radar_metrics]
    sft_vals = np.clip(sft_vals, 0, 1).tolist() + [np.clip(sft_vals[0], 0, 1)]
    ax.plot(angles, sft_vals, 'k--', lw=1.5, label='SFT Exposed', alpha=0.5)

# AAU methods
for name in AAU_ORDER:
    if name not in aau_data:
        continue
    vals = [fn(aau_data[name]) for _, fn in radar_metrics]
    vals = np.clip(vals, 0, 1).tolist() + [np.clip(vals[0], 0, 1)]
    color = AAU_COLORS[name]
    ax.plot(angles, vals, lw=2, color=color, label=name)
    ax.fill(angles, vals, alpha=0.05, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylim(0, 1)
ax.set_title('AAU-PII — Radar Chart (all axes: higher = better)',
             fontsize=12, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ======================================================================
# SUMMARY — AAU-PII vs Baselines (Pareto scatter combined)
# ======================================================================

BASELINE_COLORS = {
    'SFT Exposed':      '#7f7f7f',
    'GA ep3':           '#bcbd22',
    'GA+Retain ep3':    '#17becf',
    'GA+Retain ep5':    '#9edae5',
    'Task Vector α=2':  '#e377c2',
    'NPO rw=2 ep3':     '#f7b6d2',
    'NPO rw=1 ep5':     '#8c564b',
    'NPO rw=2 ep5':     '#c49c94',
}

fig, ax = plt.subplots(figsize=(11, 7))

# Plot baselines (gray markers)
for name in BASELINE_ORDER:
    if name not in baseline_data:
        continue
    m = baseline_data[name]
    x = 1 - m.get('PII Leakage Forget', np.nan)
    y = m.get('Model Utility', np.nan)
    color = BASELINE_COLORS.get(name, '#999999')
    ax.scatter(x, y, color=color, marker='s', s=90, zorder=4,
               edgecolors='black', linewidths=0.5, alpha=0.7)
    ax.annotate(name, (x, y), textcoords='offset points', xytext=(5, -10),
                fontsize=7, color=color, alpha=0.85)

# Plot AAU methods (larger, solid markers)
for name in AAU_ORDER:
    if name not in aau_data:
        continue
    m = aau_data[name]
    x = 1 - m.get('PII Leakage Forget', np.nan)
    y = m.get('Model Utility', np.nan)
    ax.scatter(x, y, color=AAU_COLORS[name], marker=AAU_MARKERS[name],
               s=160, zorder=6, edgecolors='black', linewidths=0.8)
    ax.annotate(name, (x, y), textcoords='offset points', xytext=(6, 5),
                fontsize=8, color=AAU_COLORS[name], fontweight='bold')

# SFT reference lines
if 'SFT Exposed' in baseline_data:
    sft_u = baseline_data['SFT Exposed'].get('Model Utility')
    sft_l = baseline_data['SFT Exposed'].get('PII Leakage Forget')
    if sft_u: ax.axhline(sft_u, color='gray', ls='--', lw=1, alpha=0.5)
    if sft_l: ax.axvline(1 - sft_l, color='gray', ls=':', lw=1, alpha=0.5)

# Legend proxies
aau_patch = mpatches.Patch(color='#1f77b4', label='AAU-PII (proposed)')
base_patch = mpatches.Patch(color='#999999', label='Baseline methods', alpha=0.7)
ax.legend(handles=[aau_patch, base_patch], fontsize=9, loc='lower left')

ax.set_xlabel('Forget Quality  (1 − PII Leakage Forget, higher = better)', fontsize=11)
ax.set_ylabel('Model Utility (higher = better)', fontsize=11)
ax.set_title('AAU-PII vs Baseline Methods — Pareto Frontier',
             fontsize=13, fontweight='bold')

# Annotate ideal corner
ax.text(0.97, 0.97, '★ Ideal', transform=ax.transAxes,
        ha='right', va='top', fontsize=9, color='green', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
# ======================================================================
# SUMMARY TABLE — AAU-PII vs Key Baselines
# ======================================================================

# Select key representatives
compare_methods = [
    # Baselines
    ('SFT Exposed',      baseline_data),
    ('GA ep3',           baseline_data),
    ('GA+Retain ep3',    baseline_data),
    ('GA+Retain ep5',    baseline_data),
    ('Task Vector α=2',  baseline_data),
    ('NPO rw=2 ep3',     baseline_data),
    ('NPO rw=2 ep5',     baseline_data),
    # AAU-PII proposed
    ('AAU-GA ep3',       aau_data),
    ('AAU-GA ep5 v2',    aau_data),
    ('AAU-NPO v1',       aau_data),
    ('AAU-NPO v2',       aau_data),
    ('AAU-TV Round3',    aau_data),
    ('AAU-TV Final',     aau_data),
]

cmp_metrics = [
    'PII Leakage Forget',
    'PII Leakage Forget Rephrase',
    'PII Leakage Forget Inverse',
    'Targeted Extraction Forget ESR',
    'Model Utility',
    'Prob. Retain',
    'Truth Ratio Retain',
]

rows = {}
for (name, src) in compare_methods:
    if name in src:
        rows[name] = {k: src[name].get(k, np.nan) for k in cmp_metrics}

cmp_df = pd.DataFrame(rows).T

# Display
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

print('=' * 120)
print('SUMMARY: AAU-PII vs Baseline Methods')
print('=' * 120)
print(cmp_df.to_string())
print('=' * 120)
print('↓ lower = better: PII Leakage, ESR')
print('↑ higher = better: Model Utility, Prob. Retain, Truth Ratio Retain')

In [ ]:
# ======================================================================
# HEATMAP — Summary comparison AAU vs Baselines
# ======================================================================

fig = make_heatmap(cmp_df, 'AAU-PII vs Baselines — Summary Heatmap (green = better)')
plt.tight_layout()
plt.show()

In [ ]:
# ======================================================================
# AUTO-GENERATED FINDINGS
# ======================================================================

print('=' * 65)
print('AAU-PII FINDINGS SUMMARY')
print('=' * 65)

# Best AAU by PII Leakage
aau_pii = {n: aau_data[n].get('PII Leakage Forget', np.nan)
           for n in AAU_ORDER if n in aau_data}
best_pii_name = min(aau_pii, key=aau_pii.get)
print(f'\n[Forget] Best PII Leakage: {best_pii_name}')
print(f'         PII Leak = {aau_pii[best_pii_name]:.4f}, '
      f'Utility = {aau_data[best_pii_name].get("Model Utility", np.nan):.4f}')

# Best AAU by Utility
aau_util = {n: aau_data[n].get('Model Utility', np.nan)
            for n in AAU_ORDER if n in aau_data}
best_util_name = max(aau_util, key=aau_util.get)
print(f'\n[Utility] Best Model Utility: {best_util_name}')
print(f'          Utility = {aau_util[best_util_name]:.4f}, '
      f'PII Leak = {aau_data[best_util_name].get("PII Leakage Forget", np.nan):.4f}')

# Best AAU by ESR
aau_esr = {n: aau_data[n].get('Targeted Extraction Forget ESR', np.nan)
           for n in AAU_ORDER if n in aau_data}
best_esr_name = min(aau_esr, key=aau_esr.get)
print(f'\n[ESR] Best Targeted ESR: {best_esr_name}')
print(f'      ESR = {aau_esr[best_esr_name]:.4f}')

# Compare best AAU vs best baseline
if 'GA+Retain ep3' in baseline_data:
    b = baseline_data['GA+Retain ep3']
    a = aau_data.get('AAU-GA ep5 v2', {})
    print(f'\n[vs GA+Retain ep3 baseline]')
    delta_pii  = (b.get('PII Leakage Forget', 0) - a.get('PII Leakage Forget', 0))
    delta_util = (a.get('Model Utility', 0) - b.get('Model Utility', 0))
    delta_esr  = (b.get('Targeted Extraction Forget ESR', 0) - a.get('Targeted Extraction Forget ESR', 0))
    print(f'  AAU-GA ep5 v2 vs GA+Retain ep3:')
    print(f'  PII Leak: {b.get("PII Leakage Forget"):.4f} → {a.get("PII Leakage Forget"):.4f} ({delta_pii:+.4f} improvement)')
    print(f'  Utility:  {b.get("Model Utility"):.4f} → {a.get("Model Utility"):.4f} ({delta_util:+.4f})')
    print(f'  ESR:      {b.get("Targeted Extraction Forget ESR"):.4f} → {a.get("Targeted Extraction Forget ESR"):.4f} ({delta_esr:+.4f} improvement)')

print('\n' + '=' * 65)

In [ ]:
# ======================================================================
# EXPORT
# ======================================================================

EXPORT_DIR = os.path.join(BASE_DIR, 'analysis_outputs')
os.makedirs(EXPORT_DIR, exist_ok=True)

# AAU-only metrics
aau_df.to_csv(os.path.join(EXPORT_DIR, 'aau_pii_metrics.csv'))
print(f'[OK] AAU metrics → {EXPORT_DIR}/aau_pii_metrics.csv')

# Full comparison
cmp_df.to_csv(os.path.join(EXPORT_DIR, 'aau_vs_baselines_summary.csv'))
print(f'[OK] Summary comparison → {EXPORT_DIR}/aau_vs_baselines_summary.csv')